## 1. CARGA DE PARÁMETROS (Con valores por defecto para evitar errores)

In [0]:
import traceback
from pyspark.sql.functions import col, upper, trim, to_date, lit
from pyspark.sql.types import DecimalType, DateType

try:
    dbutils.widgets.text("catalog", "iowa_sales", "Catalog Name")
    dbutils.widgets.text("bronze_schema", "sales_bronze", "Bronze Schema Name")
    dbutils.widgets.text("silver_schema", "sales_silver", "Silver Schema Name")
    dbutils.widgets.text("bronze_table", "sales_raw", "Bronze Table Name")
    dbutils.widgets.text("silver_table", "sales_cleaned", "Silver Table Name")
    
    env = {
        "catalog": dbutils.widgets.get("catalog"),
        "sch_bronze": dbutils.widgets.get("bronze_schema"),
        "sch_silver": dbutils.widgets.get("silver_schema"),
        "tb_bronze": dbutils.widgets.get("bronze_table"),
        "tb_silver": dbutils.widgets.get("silver_table")
    }
    
    source_table = f"{env['catalog']}.{env['sch_bronze']}.{env['tb_bronze']}"
    target_table = f"{env['catalog']}.{env['sch_silver']}.{env['tb_silver']}"
    
    print(f"Configuración cargada. \nOrigen: {source_table} \nDestino: {target_table}")

except Exception as e:
    print("Error crítico al obtener los parámetros del notebook.")
    raise e

## 2. AUDITORÍA Y REGLAS DE CALIDAD

In [0]:
from pyspark.sql.functions import col, current_timestamp, lit, expr

try:
    print(f"Leyendo datos crudos (Strings) desde la capa Bronze...")
    df_sales_raw = spark.table(source_table)
    initial_row_count = df_sales_raw.count()
    
    # -------------------------------------------------------------------------
    # VALIDACIONES Y CASTEO
    # -------------------------------------------------------------------------
    expected_doubles_to_int = ["store", "county_number", "category", "vendor_no", "itemno", "pack", "sale_bottles"]
    expected_money_and_volume = ["bottle_volume_ml", "state_bottle_cost", "state_bottle_retail", "sale_dollars", "sale_liters", "sale_gallons"]
    expected_strings = ["name", "city", "im_desc"] 

    is_corrupted_cond = lit(False)

    def clean_num_str(column_name):
        return f"REPLACE(REPLACE(REPLACE(TRIM({column_name}), '$', ''), '.', ''), ',', '.')"

    date_parsing_expr = "COALESCE(try_to_timestamp(date, 'MM/dd/yyyy'), try_to_timestamp(date, 'yyyy-MM-dd'))"

    # Evaluamos si la data no cumple con los tipos esperados
    for c in expected_doubles_to_int:
        is_corrupted_cond = is_corrupted_cond | (col(c).isNotNull() & expr(f"try_cast(try_cast({c} AS DOUBLE) AS INT) IS NULL"))

    for c in expected_money_and_volume:
        is_corrupted_cond = is_corrupted_cond | (col(c).isNotNull() & expr(f"try_cast({clean_num_str(c)} AS DOUBLE) IS NULL"))

    for c in expected_strings:
        is_corrupted_cond = is_corrupted_cond | (col(c).isNotNull() & expr(f"try_cast({c} AS DOUBLE) IS NOT NULL"))

    is_corrupted_cond = is_corrupted_cond | (col("date").isNotNull() & expr(f"{date_parsing_expr} IS NULL"))

    # Aplicamos la bandera de corrupción
    df_audited = df_sales_raw.withColumn("is_corrupted", is_corrupted_cond)

    # -------------------------------------------------------------------------
    # SEPARACIÓN: CUARENTENA VS DATOS VÁLIDOS
    # -------------------------------------------------------------------------
    critical_cols = [
        'invoice_line_no', 'date', "store", "category", "vendor_no", 
        "itemno", "pack", "bottle_volume_ml", "state_bottle_cost", 
        "state_bottle_retail", "sale_bottles"
    ]
    
    quarantine_cond = (col("is_corrupted") == True)
    for c in critical_cols:
        quarantine_cond = quarantine_cond | col(c).isNull()
        
    df_rejected = df_audited.filter(quarantine_cond).withColumn("rejection_reason", lit("Corrupted or Missing Critical Data"))
    df_valid = df_audited.filter(~quarantine_cond)

except Exception as e:
    print("Error durante la auditoría de calidad.")
    raise e

## 3. TRANSFORMACIÓN FINAL 

In [0]:
try:
    print("Aplicando casteos, limpieza y deduplicación a registros válidos...")
    
    # 1. Aplicamos casteos sobre df_valid (ahora sabemos que no fallarán porque pasaron la validación)
    for c in expected_doubles_to_int:
        df_valid = df_valid.withColumn(c, expr(f"try_cast(try_cast({c} AS DOUBLE) AS INT)"))

    for c in expected_money_and_volume:
        df_valid = df_valid.withColumn(c, expr(f"try_cast({clean_num_str(c)} AS DOUBLE)"))
        
    df_valid = df_valid.withColumn("date", expr(date_parsing_expr))
    df_valid = df_valid.withColumn("zipcode", expr("cast(try_cast(try_cast(zipcode AS DOUBLE) AS INT) AS STRING)"))

    # 2. Pipeline de limpieza final
    df_silver = (df_valid
        .dropDuplicates(["invoice_line_no"])
        .na.fill({
            "address": "UNKNOWN",
            "city": "UNKNOWN",
            "zipcode": "00000", 
            "county": "UNKNOWN",
            "category_name": "UNCATEGORIZED",
            "vendor_name": "UNKNOWN",
            "im_desc": "NO DESCRIPTION",
            "county_number": -1
        })
        .withColumn("name", upper(trim(col("name"))))
        .withColumn("address", upper(trim(col("address"))))
        .withColumn("city", upper(trim(col("city"))))
        .withColumn("county", upper(trim(col("county"))))
        .withColumn("category_name", upper(trim(col("category_name"))))
        .withColumn("vendor_name", upper(trim(col("vendor_name"))))
        .withColumn("im_desc", upper(trim(col("im_desc"))))
        .filter(
            (col("state_bottle_cost") > 0) & 
            (col("state_bottle_retail") > 0) &
            (col("bottle_volume_ml") > 0) &
            (col("sale_bottles") > 0)
        )
        .withColumn("date", to_date(col("date")))
        .withColumn("state_bottle_cost", col("state_bottle_cost").cast(DecimalType(10, 2)))
        .withColumn("state_bottle_retail", col("state_bottle_retail").cast(DecimalType(10, 2)))
        .withColumn("sale_dollars", (col("state_bottle_retail") * col("sale_bottles")).cast(DecimalType(10, 2)))
        .withColumn("sale_liters", ((col("bottle_volume_ml") * col("sale_bottles")) / 1000).cast(DecimalType(10, 3)))
        .withColumn("sale_gallons", ((col("bottle_volume_ml") * col("sale_bottles")) / 3785.411784).cast(DecimalType(10, 3)))
        .withColumn("silver_saved_date", current_timestamp())
        .select(
            col("invoice_line_no"),
            col("date"),
            col("store").alias("store_number"),
            col("name").alias("store_name"),
            col("address"),
            col("city"),
            col("zipcode").alias("zip_code"),
            col("county_number"),
            col("county"),
            col("category").alias("category_code"),
            col("category_name"),
            col("vendor_no").alias("vendor_number"),
            col("vendor_name"),
            col("itemno").alias("item_number"),
            col("im_desc").alias("item_description"),
            col("pack"),
            col("bottle_volume_ml"),
            col("state_bottle_cost"),
            col("state_bottle_retail"),
            col("sale_bottles").alias("bottles_sold"),
            col("sale_dollars"),
            col("sale_liters").alias("volume_sold_liters"),
            col("sale_gallons").alias("volume_sold_gallons"),
            col("silver_saved_date")
        )
    )
    
    final_row_count = df_silver.count()
    rejected_count = df_rejected.count()
    
    print(f"Conteo inicial (Bronze): {initial_row_count}")
    print(f"Enviados a Cuarentena: {rejected_count}")
    print(f"Conteo final (Silver): {final_row_count}")

except Exception as e:
    print("Error durante la transformación Silver.")
    raise e

## 4. ESCRITURA INCREMENTAL DE TABLAS

In [0]:
from delta.tables import DeltaTable

try:
    
    # 1. ESCRITURA DE LA TABLA DE CUARENTENA (Modo Append - Historial de Errores)
    quarantine_table = f"{env['catalog']}.{env['sch_silver']}.sales_quarantine"
    print(f"Agregando registros rechazados en {quarantine_table} (DLQ)...")
    
    # La cuarentena es un log de auditoría, por lo que usamos Append para tener el historial.
    df_rejected.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable(quarantine_table)

    # 2. ESCRITURA DE LA TABLA SILVER (Patrón MERGE / Upsert)
    print(f"Evaluando estrategia de escritura para {target_table}...")
    
    # Verificamos si la tabla ya existe en Unity Catalog
    if spark.catalog.tableExists(target_table):
        print("La tabla existe. Ejecutando MERGE (Upsert) incremental...")
        
        # Instanciamos la tabla destino
        delta_target = DeltaTable.forName(spark, target_table)
        
        # Ejecutamos el Merge comparando la llave primaria (invoice_line_no)
        (delta_target.alias("target")
         .merge(
             df_silver.alias("source"),
             "target.invoice_line_no = source.invoice_line_no"
         )
         .whenMatchedUpdateAll() # Si ya existe y hay cambios, actualiza toda la fila
         .whenNotMatchedInsertAll() # Si no existe, la inserta como nueva
         .execute()
        )
        print("MERGE en capa Silver completado exitosamente.")
        
    else:
        print("La tabla NO existe. Ejecutando carga inicial (Overwrite)...")
        # Si es la primera vez que corremos el notebook, creamos la tabla
        df_silver.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(target_table)
            
        print("Carga inicial en capa Silver completada exitosamente.")

except Exception as e:
    print("Error al escribir las tablas finales.")
    raise e

dbutils.notebook.exit("Silver layer processing complete.")